# 05C — Sentiment Inference and Aggregation for Entity-Conditioned AI Impact Analysis

Pipeline stage:

- load the full entity-context corpus from Stage 04
- load the trained sentiment model and label mapping from Stage 05B
- rebuild the exact inference text template used in training
- run full-corpus sentiment inference
- export row-level predictions with calibrated class probabilities
- aggregate sentiment by entity, type, time, domain, and optional topic
- export coverage-filtered ranking tables for downstream reporting

Input directories

`output/04_entity_extraction/04B_analysis_entities/`

- `entity_contexts_final.parquet`

`output/05_sentiment_model/`

- `best_model/`
- `label_mapping.json`
- `training_run_config.json`

Output directory

`output/05_sentiment_inference/`

- `entity_context_sentiment_predictions.parquet`
- `entity_sentiment_summary.parquet`
- `entity_sentiment_summary.csv`
- `entity_type_sentiment_summary.parquet`
- `entity_type_sentiment_summary.csv`
- `entity_time_sentiment_summary.parquet`
- `entity_time_sentiment_summary.csv`
- `domain_sentiment_summary.parquet`
- `domain_sentiment_summary.csv`
- `topic_sentiment_summary.parquet`
- `topic_sentiment_summary.csv`
- `top_positive_companies_display.csv`
- `top_negative_companies_display.csv`
- `top_positive_technologies_display.csv`
- `top_negative_technologies_display.csv`
- `sentiment_inference_run_config.json`
- `sentiment_inference_report.json`

In [1]:
!pip -q install pyarrow==17.0.0 transformers==4.45.1 accelerate==0.34.2

## Environment Setup

In [2]:
import os
import gc
import re
import json
import hashlib
import random
import warnings
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModelForSequenceClassification, AutoTokenizer

from google.colab import drive
drive.mount("/content/drive")

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"

IN_DIR_04B = f"{BASE_DIR}/output/04_entity_extraction/04B_analysis_entities"
IN_DIR_05B = f"{BASE_DIR}/output/05_sentiment_model"
OUT_DIR_05C = f"{BASE_DIR}/output/05_sentiment_inference"

ENTITY_CONTEXTS_FINAL_PARQUET = f"{IN_DIR_04B}/entity_contexts_final.parquet"
MODEL_DIR = f"{IN_DIR_05B}/best_model"
LABEL_MAPPING_JSON = f"{IN_DIR_05B}/label_mapping.json"
TRAINING_RUN_CONFIG_JSON = f"{IN_DIR_05B}/training_run_config.json"

OPTIONAL_TOPIC_PATHS = [
    f"{BASE_DIR}/output/04/docs_with_topics.parquet",
    f"{BASE_DIR}/output/03_topic_modeling/docs_with_topics.parquet",
    f"{BASE_DIR}/output/03_topic_modeling/04_docs_with_topics.parquet",
]

ROW_LEVEL_PREDICTIONS_PARQUET = f"{OUT_DIR_05C}/entity_context_sentiment_predictions.parquet"

ENTITY_SUMMARY_PARQUET = f"{OUT_DIR_05C}/entity_sentiment_summary.parquet"
ENTITY_SUMMARY_CSV = f"{OUT_DIR_05C}/entity_sentiment_summary.csv"

ENTITY_TYPE_SUMMARY_PARQUET = f"{OUT_DIR_05C}/entity_type_sentiment_summary.parquet"
ENTITY_TYPE_SUMMARY_CSV = f"{OUT_DIR_05C}/entity_type_sentiment_summary.csv"

ENTITY_TIME_SUMMARY_PARQUET = f"{OUT_DIR_05C}/entity_time_sentiment_summary.parquet"
ENTITY_TIME_SUMMARY_CSV = f"{OUT_DIR_05C}/entity_time_sentiment_summary.csv"

DOMAIN_SUMMARY_PARQUET = f"{OUT_DIR_05C}/domain_sentiment_summary.parquet"
DOMAIN_SUMMARY_CSV = f"{OUT_DIR_05C}/domain_sentiment_summary.csv"

TOPIC_SUMMARY_PARQUET = f"{OUT_DIR_05C}/topic_sentiment_summary.parquet"
TOPIC_SUMMARY_CSV = f"{OUT_DIR_05C}/topic_sentiment_summary.csv"

TOP_POSITIVE_COMPANIES_DISPLAY_CSV = f"{OUT_DIR_05C}/top_positive_companies_display.csv"
TOP_NEGATIVE_COMPANIES_DISPLAY_CSV = f"{OUT_DIR_05C}/top_negative_companies_display.csv"
TOP_POSITIVE_TECHNOLOGIES_DISPLAY_CSV = f"{OUT_DIR_05C}/top_positive_technologies_display.csv"
TOP_NEGATIVE_TECHNOLOGIES_DISPLAY_CSV = f"{OUT_DIR_05C}/top_negative_technologies_display.csv"

INFERENCE_RUN_CONFIG_JSON = f"{OUT_DIR_05C}/sentiment_inference_run_config.json"
INFERENCE_REPORT_JSON = f"{OUT_DIR_05C}/sentiment_inference_report.json"

os.makedirs(OUT_DIR_05C, exist_ok=True)

assert os.path.exists(ENTITY_CONTEXTS_FINAL_PARQUET), f"Missing: {ENTITY_CONTEXTS_FINAL_PARQUET}"
assert os.path.exists(MODEL_DIR), f"Missing: {MODEL_DIR}"
assert os.path.exists(LABEL_MAPPING_JSON), f"Missing: {LABEL_MAPPING_JSON}"

print("ENTITY_CONTEXTS_FINAL_PARQUET:", ENTITY_CONTEXTS_FINAL_PARQUET)
print("MODEL_DIR:", MODEL_DIR)
print("LABEL_MAPPING_JSON:", LABEL_MAPPING_JSON)
print("TRAINING_RUN_CONFIG_JSON exists:", os.path.exists(TRAINING_RUN_CONFIG_JSON))
print("OUT_DIR_05C:", OUT_DIR_05C)

ENTITY_CONTEXTS_FINAL_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_contexts_final.parquet
MODEL_DIR: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/best_model
LABEL_MAPPING_JSON: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/label_mapping.json
TRAINING_RUN_CONFIG_JSON exists: True
OUT_DIR_05C: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference


## Runtime and Configuration

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEFAULT_VALID_ENTITY_TYPES = {"company", "technology", "government_institution", "person"}

if os.path.exists(TRAINING_RUN_CONFIG_JSON):
    with open(TRAINING_RUN_CONFIG_JSON, "r", encoding="utf-8") as f:
        training_run_config = json.load(f)
else:
    training_run_config = {}

MODEL_NAME_FOR_REFERENCE = training_run_config.get("model_name", "best_model")
MAX_CONTEXT_CHARS = int(training_run_config.get("max_context_chars", 1400))
MAX_LENGTH = int(training_run_config.get("max_length", 384))
VALID_ENTITY_TYPES = set(training_run_config.get("valid_entity_types", sorted(DEFAULT_VALID_ENTITY_TYPES)))
USE_ENTITY_CONDITIONED_INPUT = bool(training_run_config.get("use_entity_conditioned_input", True))
TEXT_TEMPLATE_VERSION = str(training_run_config.get("text_template_version", "v2_entity_conditioned"))

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0).lower()
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    if "h100" in gpu_name:
        INFER_BATCH_SIZE = 1024
        NUM_WORKERS = 8
    elif total_mem_gb >= 70:
        INFER_BATCH_SIZE = 768
        NUM_WORKERS = 8
    elif total_mem_gb >= 40:
        INFER_BATCH_SIZE = 512
        NUM_WORKERS = 6
    elif total_mem_gb >= 20:
        INFER_BATCH_SIZE = 256
        NUM_WORKERS = 4
    else:
        INFER_BATCH_SIZE = 128
        NUM_WORKERS = 2
else:
    gpu_name = "cpu"
    total_mem_gb = 0.0
    INFER_BATCH_SIZE = 32
    NUM_WORKERS = 2

USE_BF16 = torch.cuda.is_available()
USE_FP16 = False

MIN_CONTEXT_CHARS = 40
MIN_ENTITY_LEN = 2

JOIN_TOPICS_IF_AVAILABLE = True

MIN_DOCS_FOR_DISPLAY = 10
MIN_ROWS_FOR_DISPLAY = 15
TOP_DISPLAY_N = 30

# 05C input-side deterministic filter
APPLY_PREINFERENCE_ENTITY_FILTER = True

ALLOWED_GOVERNMENT_ENTITIES = {
    "united states",
    "us",
    "u.s.",
    "u.s",
    "usa",
    "china",
    "india",
    "united kingdom",
    "uk",
    "u.k.",
    "u.k",
    "european union",
    "eu",
    "california",
    "new york",
    "san francisco",
    "japan",
    "germany",
    "france",
    "canada",
    "south korea",
    "singapore",
    "taiwan",
    "white house",
    "u.s. congress",
    "us congress",
    "congress",
    "federal communications commission",
    "fcc",
    "european commission",
    "ministry of industry and information technology",
}

GENERIC_GOVERNMENT_TERMS = {
    "world",
    "global",
    "international",
    "europe",
    "asia",
    "africa",
    "america",
    "americas",
}

DISPLAY_EXCLUDE_PATTERNS = [
    r"^\s*#",
    r"^\s*\$",
    r"\bbooth\b",
    r"\bstand\b",
    r"\bhall\b",
    r"\bsuite\b",
    r"\btoken\b",
    r"\bmarket\b",
    r"\bvaluation\b",
    r"\bvalued company\b",
    r"\bstartup\b",
    r"\bfintech\b",
    r"\d{3,}",
]

run_config = {
    "seed": SEED,
    "model_name_for_reference": MODEL_NAME_FOR_REFERENCE,
    "max_context_chars": MAX_CONTEXT_CHARS,
    "max_length": MAX_LENGTH,
    "valid_entity_types": sorted(VALID_ENTITY_TYPES),
    "use_entity_conditioned_input": USE_ENTITY_CONDITIONED_INPUT,
    "text_template_version": TEXT_TEMPLATE_VERSION,
    "infer_batch_size": INFER_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "use_bf16": USE_BF16,
    "use_fp16": USE_FP16,
    "min_context_chars": MIN_CONTEXT_CHARS,
    "min_entity_len": MIN_ENTITY_LEN,
    "join_topics_if_available": JOIN_TOPICS_IF_AVAILABLE,
    "min_docs_for_display": MIN_DOCS_FOR_DISPLAY,
    "min_rows_for_display": MIN_ROWS_FOR_DISPLAY,
    "top_display_n": TOP_DISPLAY_N,
    "apply_preinference_entity_filter": APPLY_PREINFERENCE_ENTITY_FILTER,
    "allowed_government_entities": sorted(ALLOWED_GOVERNMENT_ENTITIES),
    "generic_government_terms": sorted(GENERIC_GOVERNMENT_TERMS),
    "display_exclude_patterns": DISPLAY_EXCLUDE_PATTERNS,
}

with open(INFERENCE_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

print("wrote:", INFERENCE_RUN_CONFIG_JSON)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("gpu total memory (GB):", round(total_mem_gb, 2))
print("INFER_BATCH_SIZE:", INFER_BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/sentiment_inference_run_config.json
torch.cuda.is_available(): True
cuda device: NVIDIA RTX PRO 6000 Blackwell Server Edition
gpu total memory (GB): 94.97
INFER_BATCH_SIZE: 768
NUM_WORKERS: 8


## Utility Functions

In [5]:
WS_RE = re.compile(r"\s+")

def normalize_spaces(text: str) -> str:
    return WS_RE.sub(" ", str(text or "")).strip()

def truncate_text(text: str, max_chars: int) -> str:
    text = normalize_spaces(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

def sha1_text(text: str) -> str:
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()

def safe_json_dump(obj: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def find_optional_topic_file(candidate_paths: List[str]) -> Optional[str]:
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    return None

def build_row_uid(df_in: pd.DataFrame) -> pd.Series:
    group_idx = (
        df_in.groupby(["doc_id", "block_id", "canonical_entity", "final_type"], dropna=False)
        .cumcount()
        .astype(str)
    )
    return (
        df_in["doc_id"].astype(str)
        + "||"
        + df_in["block_id"].astype(str)
        + "||"
        + df_in["canonical_entity"].astype(str)
        + "||"
        + df_in["final_type"].astype(str)
        + "||"
        + group_idx
    )

def normalize_entity_key(text: str) -> str:
    text = normalize_spaces(text).lower()
    text = re.sub(r"[’']", "", text)
    text = re.sub(r"[\(\)\[\]\{\}\.,;:!?\"“”‘’/\\|]+", " ", text)
    text = re.sub(r"[-_]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_title_case(text: str) -> str:
    words = []
    for w in str(text or "").split():
        if re.fullmatch(r"[A-Z]{2,6}", w):
            words.append(w)
        elif re.fullmatch(r"[A-Za-z]+", w):
            words.append(w.capitalize())
        else:
            words.append(w)
    return " ".join(words)

def canonicalize_surface_05c(text: str) -> str:
    s = normalize_spaces(text)

    # strip frequent outer noise but keep meaningful inner symbols like 01.ai
    s = re.sub(r"^[\+\*\|~]+", "", s)
    s = re.sub(r"^[\.:;,_\- ]+", "", s)
    s = re.sub(r"[\.:;,_\- ]+$", "", s)

    # remove concatenated leading time-like fragments: 06Press -> Press
    s = re.sub(r"^\d{2}(?=[A-Z][a-z])", "", s)

    # remove duplicated leading + with/without space
    s = re.sub(r"^\+\s*", "", s)

    s = normalize_spaces(s)

    safe_map = {
        "open ai": "OpenAI",
        "chat gpt": "ChatGPT",
        "gpt 4": "GPT-4",
        "gpt4": "GPT-4",
        "gpt 4o": "GPT-4o",
        "gpt4o": "GPT-4o",
        "u s": "United States",
        "u s a": "United States",
        "usa": "United States",
        "u k": "United Kingdom",
        "uk": "United Kingdom",
        "eu": "European Union",
        "a i": "AI",
        "ai": "AI",
        "llms": "LLMs",
        "llm": "LLM",
        "0g labs": "0G Labs",
        "0g foundation": "0G Foundation",
        "01.ai": "01.ai",
        "01 ai": "01.ai",
    }

    key = normalize_entity_key(s)
    if key in safe_map:
        return safe_map[key]

    if re.fullmatch(r"[A-Z]{2,8}", s):
        return s

    return simple_title_case(s)

def is_numeric_or_price_like_entity(text: str) -> bool:
    s = normalize_spaces(text)
    if s == "":
        return True
    if re.fullmatch(r"[#\$€£¥]?\d[\d,.\-+%/ ]*", s):
        return True
    if re.fullmatch(r"[$€£¥]\s*\d[\d,.\-+% ]*", s):
        return True
    if re.fullmatch(r"[#\$€£¥]?[A-Z]*\d+[A-Z0-9\-+./% ]*", s):
        return True
    if re.search(r"^\s*[$€£¥#]", s):
        return True
    return False

def has_catalog_or_promo_shape(text: str) -> bool:
    s = normalize_spaces(text)
    patterns = [
        r"^\s*#",
        r"^\s*\$",
        r"\bbooth\b",
        r"\bstand\b",
        r"\bhall\b",
        r"\bsuite\b",
        r"\bpresale\b",
        r"\btoken\b",
        r"\bmarket\b",
        r"\bvaluation\b",
        r"\bvalued company\b",
        r"\bstartup\b",
        r"\bfintech\b",
        r"\bseries [a-z]\b",
        r"\d{4,}",
    ]
    return any(re.search(p, s, flags=re.IGNORECASE) for p in patterns)

def is_fragmentary_person(text: str, final_type: str) -> bool:
    if final_type != "person":
        return False
    s = normalize_spaces(text)
    toks = s.split()
    if len(toks) == 1:
        tok = toks[0]
        if len(tok) <= 2:
            return True
        if re.fullmatch(r"[A-Z][a-z]+", tok):
            return True
    return False

def is_generic_or_low_value_technology(text: str, final_type: str) -> bool:
    if final_type != "technology":
        return False

    key = normalize_entity_key(text)

    generic_terms = {
        "ai driven search engine",
        "ai music generators",
        "ai generated robocall",
        "gaming algorithms",
        "algorithm black box",
        "human like chatbot personas",
        "government social scoring systems",
        "ai report writing software",
        "facial recognition systems",
        "live facial scanning",
        "0 code low code architecture",
        "01 family",
        "01 mini models",
        "01 model",
        "01 models",
        "01 reasoning model",
    }

    if key in generic_terms:
        return True

    if re.fullmatch(r"0?\d+\s+(family|model|models|reasoning model)", key):
        return True

    if key.startswith("ai ") and len(key.split()) >= 3:
        return True

    return False

def is_low_value_government_entity(text: str, final_type: str) -> bool:
    if final_type != "government_institution":
        return False

    key = normalize_entity_key(text)

    allowed = {
        "united states",
        "us",
        "u s",
        "usa",
        "china",
        "india",
        "united kingdom",
        "uk",
        "european union",
        "eu",
        "california",
        "new york",
        "san francisco",
        "japan",
        "germany",
        "france",
        "canada",
        "south korea",
        "singapore",
        "taiwan",
        "white house",
        "us congress",
        "u s congress",
        "congress",
        "federal communications commission",
        "fcc",
        "european commission",
    }
    generic = {
        "world",
        "global",
        "international",
        "europe",
        "asia",
        "africa",
        "america",
        "americas",
    }

    if key in allowed:
        return False
    if key in generic:
        return True
    if is_numeric_or_price_like_entity(text):
        return True
    if has_catalog_or_promo_shape(text):
        return True
    return False

def should_keep_inference_row(row: pd.Series) -> bool:
    entity = normalize_spaces(row.get("canonical_entity", ""))
    final_type = normalize_spaces(row.get("final_type", ""))

    if entity == "":
        return False
    if len(entity) < MIN_ENTITY_LEN:
        return False
    if is_numeric_or_price_like_entity(entity):
        return False
    if has_catalog_or_promo_shape(entity):
        return False
    if is_fragmentary_person(entity, final_type):
        return False
    if is_low_value_government_entity(entity, final_type):
        return False
    if is_generic_or_low_value_technology(entity, final_type):
        return False
    return True

def build_model_input(row: pd.Series) -> str:
    entity = normalize_spaces(row.get("canonical_entity", ""))
    entity_type = normalize_spaces(row.get("final_type", ""))
    title = normalize_spaces(row.get("title", ""))
    domain = normalize_spaces(row.get("domain", ""))
    context = truncate_text(row.get("context_text", ""), MAX_CONTEXT_CHARS)

    if USE_ENTITY_CONDITIONED_INPUT:
        parts = [
            f"Target entity: {entity}",
            f"Entity type: {entity_type}",
        ]
        if title:
            parts.append(f"Document title: {title}")
        if domain:
            parts.append(f"Source domain: {domain}")
        parts.append(
            "Instruction: classify the sentiment of AI-related impact on the target entity "
            "in this context as positive, negative, or mixed_or_unclear."
        )
        parts.append(f"Context: {context}")
        return "\n".join(parts)

    return context

def aggregate_sentiment(df_in: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    base = (
        df_in.groupby(group_cols, dropna=False)
        .agg(
            n_rows=("row_uid", "size"),
            n_docs=("doc_id", "nunique"),
            n_domains=("domain", "nunique"),
            mention_confidence_mean=("mention_confidence", "mean"),
            pred_confidence_mean=("pred_confidence", "mean"),
            prob_negative_mean=("prob_negative", "mean"),
            prob_mixed_or_unclear_mean=("prob_mixed_or_unclear", "mean"),
            prob_positive_mean=("prob_positive", "mean"),
        )
        .reset_index()
    )

    counts = (
        df_in.groupby(group_cols + ["pred_sentiment_label"], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["mixed_or_unclear", "negative", "positive"]:
        if col not in counts.columns:
            counts[col] = 0

    out = base.merge(counts, on=group_cols, how="left")
    out["hard_label_mode"] = out[["negative", "mixed_or_unclear", "positive"]].idxmax(axis=1)
    out["sentiment_index"] = out["prob_positive_mean"] - out["prob_negative_mean"]
    out["directional_confidence"] = 1.0 - out["prob_mixed_or_unclear_mean"]
    out["impact_score"] = out["sentiment_index"] * out["directional_confidence"] * np.log1p(out["n_docs"])
    out["abs_sentiment_index"] = out["sentiment_index"].abs()
    return out

def export_table(df_out: pd.DataFrame, parquet_path: str, csv_path: str) -> None:
    df_out.to_parquet(parquet_path, index=False)
    df_out.to_csv(csv_path, index=False)
    print("wrote:", parquet_path)
    print("wrote:", csv_path)

def apply_display_filter(df_in: pd.DataFrame) -> pd.DataFrame:
    if len(df_in) == 0:
        return df_in.copy()

    out = df_in.copy()
    if "canonical_entity" not in out.columns:
        return out

    entity_key = out["canonical_entity"].fillna("").astype(str).map(normalize_entity_key)

    bad_patterns = [
        r"^\s*#",
        r"^\s*\$",
        r"^\s*\+",
        r"^\s*\d{2}(?=[A-Z][a-z])",
        r"\bbooth\b",
        r"\btoken\b",
        r"\bpresale\b",
        r"\bmarket\b",
        r"\bstartup\b",
        r"\bfintech\b",
        r"\bvalued company\b",
        r"\d{4,}",
    ]

    mask = pd.Series(True, index=out.index)
    text_series = out["canonical_entity"].fillna("").astype(str)

    for pattern in bad_patterns:
        mask &= ~text_series.str.contains(pattern, case=False, regex=True)

    generic_display_keys = {
        "ai driven search engine",
        "ai music generators",
        "ai generated robocall",
        "gaming algorithms",
        "algorithm black box",
        "human like chatbot personas",
        "government social scoring systems",
        "ai report writing software",
        "facial recognition systems",
        "live facial scanning",
        "01 family",
        "01 mini models",
        "01 model",
        "01 models",
        "01 reasoning model",
        "tessa",
    }

    mask &= ~entity_key.isin(generic_display_keys)
    return out[mask].copy()

## Load Entity Contexts for Inference

In [6]:
df = pd.read_parquet(ENTITY_CONTEXTS_FINAL_PARQUET).copy()
print("entity_contexts_final raw shape:", df.shape)
display(df.head(5))

required_cols = {
    "doc_id",
    "block_id",
    "date",
    "domain",
    "title",
    "canonical_entity",
    "final_type",
}
missing_cols = required_cols - set(df.columns)
assert not missing_cols, f"Missing required columns: {missing_cols}"

if "clean_block_text" in df.columns:
    text_source_col = "clean_block_text"
elif "context_text" in df.columns:
    text_source_col = "context_text"
else:
    raise AssertionError("Missing text column: expected clean_block_text or context_text")

df["context_text"] = df[text_source_col].fillna("").astype(str).map(normalize_spaces)
df["canonical_entity"] = df["canonical_entity"].fillna("").astype(str).map(canonicalize_surface_05c)
df["final_type"] = df["final_type"].fillna("").astype(str).map(normalize_spaces)
df["title"] = df["title"].fillna("").astype(str).map(normalize_spaces)
df["domain"] = df["domain"].fillna("").astype(str).map(normalize_spaces)
df["url"] = df["url"].fillna("").astype(str) if "url" in df.columns else ""
df["mention_confidence"] = pd.to_numeric(df.get("confidence", np.nan), errors="coerce")
df["mention_confidence"] = df["mention_confidence"].fillna(
    df["mention_confidence"].median() if df["mention_confidence"].notna().any() else 0.0
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year_month"] = df["date"].dt.to_period("M").astype(str)

df = df[df["final_type"].isin(VALID_ENTITY_TYPES)].copy()
df = df[df["canonical_entity"].str.len() >= MIN_ENTITY_LEN].copy()
df = df[df["context_text"].str.len() >= MIN_CONTEXT_CHARS].copy()

before_filter_n = len(df)
keep_mask = df.apply(should_keep_inference_row, axis=1)
removed_n = int((~keep_mask).sum())
df = df[keep_mask].copy()

df["row_uid"] = build_row_uid(df)
df = df.drop_duplicates("row_uid").reset_index(drop=True)

print("entity_contexts_final filtered shape:", df.shape)
print("rows removed by pre-inference filter:", removed_n)
print("rows retained after pre-inference filter:", len(df))

print("\nrows by final_type after filter:")
display(df["final_type"].value_counts(dropna=False))

print("\nexample inference rows after filter:")
display(
    df[
        [
            "canonical_entity",
            "final_type",
            "domain",
            "title",
            "context_text",
        ]
    ].head(20)
)

entity_contexts_final raw shape: (614017, 11)


,canonical_entity,final_type,domain,title,date,clean_block_text,confidence,doc_id,block_id,url,clean_block_len
0,#17 Mike’s Famous Philly,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.477525,47114,5,https://www.businesswire.com/news/home/2024012...,351
1,#9 Club Supreme,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.745871,47114,5,https://www.businesswire.com/news/home/2024012...,351
2,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,"Super Micro Computer, Inc.: Supermicro Unveils...",2025-06-11,Delivers high-performing AI infrastructure thr...,0.454479,153982,6,https://www.finanznachrichten.de/nachrichten-2...,171
3,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,KX Launches Its First Agentic AI Blueprint for...,2025-06-11,Delivers high-performing AI infrastructure thr...,0.454228,34586,22,https://www.finanznachrichten.de/nachrichten-2...,171
4,$101m Startup,company,cityam.com,"Musicians know AI is stealing their art, but f...",2023-11-29,Stability AI exec leaves amid concerns over ‘f...,0.634083,12444,38,https://www.cityam.com/musicians-ai-artificial...,86


entity_contexts_final filtered shape: (560998, 15)
rows removed by pre-inference filter: 51377
rows retained after pre-inference filter: 560998

rows by final_type after filter:


,count
final_type,
person,208911
technology,146631
company,131873
government_institution,73583



example inference rows after filter:


,canonical_entity,final_type,domain,title,context_text
0,SABI AI Corp,company,uppermichiganssource.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
1,SABI AI Corp,company,wdtv.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
2,SABI AI Corp,company,walb.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
3,SABI AI Corp,company,uppermichiganssource.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
4,SABI AI Corp,company,wdtv.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
5,SABI AI Corp,company,walb.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
6,01.ai,company,irishtimes.com,Chinese AI start-ups power ahead in the race t...,May 9 2024 - 05:00Four Chinese generative arti...
7,01.ai,company,nbcmiami.com,How China’s new AI model DeepSeek is threateni...,1 hour ago The No. 1 mistake people make when ...
8,01.ai,company,nbcdfw.com,How China’s new AI model DeepSeek is threateni...,45 mins ago Here's what CEOs are saying about ...
9,01AI_Yi,company,thehindu.com,AI scientist Kai-Fu Lee’s startup releases ope...,Indian engineering student builds AI model for...


## Optional Topic Join

In [7]:
topic_path = find_optional_topic_file(OPTIONAL_TOPIC_PATHS)

if JOIN_TOPICS_IF_AVAILABLE and topic_path is not None:
    topic_df = pd.read_parquet(topic_path)
    topic_keep_cols = [c for c in ["doc_id", "topic_id", "topic_label", "topic_name"] if c in topic_df.columns]

    if topic_keep_cols:
        topic_df = topic_df[topic_keep_cols].drop_duplicates("doc_id")
        df = df.merge(topic_df, on="doc_id", how="left")
        print("topic join applied from:", topic_path)
    else:
        print("topic file found but no usable topic columns detected:", topic_path)
else:
    print("topic join skipped.")

topic join skipped.


## Load Label Mapping and Model

In [8]:
with open(LABEL_MAPPING_JSON, "r", encoding="utf-8") as f:
    label_map = json.load(f)

label2id = {str(k): int(v) for k, v in label_map["label2id"].items()}
id2label = {int(k): str(v) for k, v in label_map["id2label"].items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("loaded labels:", id2label)
print("device:", device)

loaded labels: {0: 'positive', 1: 'negative', 2: 'mixed_or_unclear'}
device: cuda


## Build Model Inputs

In [9]:
df["model_input"] = df.apply(build_model_input, axis=1)
df["input_hash"] = df["model_input"].map(sha1_text)

print("example model inputs:")
display(df[["canonical_entity", "final_type", "model_input"]].head(3))

example model inputs:


,canonical_entity,final_type,model_input
0,SABI AI Corp,company,Target entity: SABI AI Corp\nEntity type: comp...
1,SABI AI Corp,company,Target entity: SABI AI Corp\nEntity type: comp...
2,SABI AI Corp,company,Target entity: SABI AI Corp\nEntity type: comp...


## Inference Dataset

In [10]:
class SentimentInferenceDataset(Dataset):
    def __init__(self, df_in: pd.DataFrame, tokenizer_obj, max_length: int):
        self.df = df_in.reset_index(drop=True)
        self.tokenizer = tokenizer_obj
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        encoded = self.tokenizer(
            row["model_input"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        return encoded

def collate_batch(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    return tokenizer.pad(
        batch,
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )

infer_dataset = SentimentInferenceDataset(df, tokenizer, MAX_LENGTH)
infer_loader = DataLoader(
    infer_dataset,
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_batch,
)

print("infer_dataset size:", len(infer_dataset))
print("infer_loader batches:", len(infer_loader))

infer_dataset size: 560998
infer_loader batches: 731


## Full-Corpus Inference

In [11]:
from tqdm.auto import tqdm

all_pred_ids: List[np.ndarray] = []
all_probs: List[np.ndarray] = []

with torch.no_grad():
    for batch in tqdm(infer_loader, total=len(infer_loader), desc="Inference"):
        batch = {k: v.to(device) for k, v in batch.items()}

        if torch.cuda.is_available() and USE_BF16:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
        elif torch.cuda.is_available() and USE_FP16:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**batch)
        else:
            outputs = model(**batch)

        logits = outputs.logits
        probs = F.softmax(logits.float(), dim=1)
        pred_ids = torch.argmax(probs, dim=1)

        all_pred_ids.append(pred_ids.detach().cpu().numpy())
        all_probs.append(probs.detach().float().cpu().numpy())

pred_ids = np.concatenate(all_pred_ids, axis=0)
pred_probs = np.concatenate(all_probs, axis=0)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("pred_ids shape:", pred_ids.shape)
print("pred_probs shape:", pred_probs.shape)

Inference:   0%|          | 0/731 [00:00<?, ?it/s]

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenize

pred_ids shape: (560998,)
pred_probs shape: (560998, 3)


## Attach Predictions

In [12]:
df_pred = df.copy()

df_pred["pred_label_id"] = pred_ids.astype(int)
df_pred["pred_sentiment_label"] = pd.Series(pred_ids).map(id2label).values

for class_id in sorted(id2label.keys()):
    label_name = id2label[class_id]
    df_pred[f"prob_{label_name}"] = pred_probs[:, class_id]

prob_cols = [f"prob_{id2label[i]}" for i in sorted(id2label.keys())]
df_pred["pred_confidence"] = df_pred[prob_cols].max(axis=1)
df_pred["pred_margin"] = (
    np.sort(pred_probs, axis=1)[:, -1] - np.sort(pred_probs, axis=1)[:, -2]
    if pred_probs.shape[1] >= 2
    else df_pred["pred_confidence"].values
)

print("prediction distribution:")
display(df_pred["pred_sentiment_label"].value_counts(dropna=False))

display(
    df_pred[
        [
            "canonical_entity",
            "final_type",
            "pred_sentiment_label",
            "pred_confidence",
            "pred_margin",
            "context_text",
        ]
    ].head(10)
)

prediction distribution:


,count
pred_sentiment_label,
positive,330451
mixed_or_unclear,172097
negative,58450


,canonical_entity,final_type,pred_sentiment_label,pred_confidence,pred_margin,context_text
0,SABI AI Corp,company,positive,0.923650,0.869887,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
1,SABI AI Corp,company,positive,0.923455,0.869493,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
2,SABI AI Corp,company,positive,0.924199,0.870823,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
3,SABI AI Corp,company,positive,0.923650,0.869887,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
4,SABI AI Corp,company,positive,0.923455,0.869493,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
5,SABI AI Corp,company,positive,0.924199,0.870823,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
6,01.ai,company,positive,0.848588,0.766833,May 9 2024 - 05:00Four Chinese generative arti...
7,01.ai,company,mixed_or_unclear,0.481885,0.012998,1 hour ago The No. 1 mistake people make when ...
8,01.ai,company,positive,0.483866,0.020351,45 mins ago Here's what CEOs are saying about ...
9,01AI_Yi,company,positive,0.914405,0.857747,Indian engineering student builds AI model for...


## Export Row-Level Predictions

In [13]:
row_keep_cols = [
    c for c in [
        "row_uid",
        "doc_id",
        "block_id",
        "date",
        "year_month",
        "domain",
        "title",
        "url",
        "canonical_entity",
        "final_type",
        "mention_confidence",
        "context_text",
        "pred_label_id",
        "pred_sentiment_label",
        "pred_confidence",
        "pred_margin",
        "prob_negative",
        "prob_mixed_or_unclear",
        "prob_positive",
        "topic_id",
        "topic_label",
        "topic_name",
    ] if c in df_pred.columns
]

df_pred[row_keep_cols].to_parquet(ROW_LEVEL_PREDICTIONS_PARQUET, index=False)
print("wrote:", ROW_LEVEL_PREDICTIONS_PARQUET)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_context_sentiment_predictions.parquet


## Aggregate by Entity

In [14]:
entity_summary = (
    aggregate_sentiment(
        df_pred,
        group_cols=["canonical_entity", "final_type"],
    )
    .sort_values(
        ["n_docs", "n_rows", "abs_sentiment_index", "pred_confidence_mean"],
        ascending=[False, False, False, False],
    )
    .reset_index(drop=True)
)

entity_summary = apply_display_filter(entity_summary).reset_index(drop=True)

export_table(entity_summary, ENTITY_SUMMARY_PARQUET, ENTITY_SUMMARY_CSV)

print("entity_summary shape:", entity_summary.shape)
print("\ntop entities by document coverage:")
display(
    entity_summary[
        (entity_summary["n_docs"] >= 20) | (entity_summary["n_rows"] >= 30)
    ].head(20)
)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_sentiment_summary.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_sentiment_summary.csv
entity_summary shape: (264291, 18)

top entities by document coverage:


,canonical_entity,final_type,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index


## Aggregate by Entity Type

In [15]:
entity_type_summary = (
    aggregate_sentiment(
        df_pred,
        group_cols=["final_type"],
    )
    .sort_values(["n_docs", "n_rows"], ascending=[False, False])
    .reset_index(drop=True)
)

export_table(entity_type_summary, ENTITY_TYPE_SUMMARY_PARQUET, ENTITY_TYPE_SUMMARY_CSV)
display(entity_type_summary)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_type_sentiment_summary.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_type_sentiment_summary.csv


,final_type,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index
0,person,208911,98601,4640,0.868538,0.845963,0.186766,0.384602,0.428632,84015,29928,94968,positive,0.241866,0.615398,1.711533,0.241866
1,company,131873,57163,3102,0.801551,0.871440,0.100937,0.215452,0.683612,27313,6252,98308,positive,0.582675,0.784548,5.007331,0.582675
2,technology,146631,55619,3422,0.713391,0.859716,0.130310,0.239508,0.630182,33321,11711,101599,positive,0.499872,0.760492,4.153617,0.499872
3,government_institution,73583,40753,3652,0.741925,0.838197,0.190242,0.369079,0.440680,27448,10559,35576,positive,0.250438,0.630921,1.677289,0.250438


## Aggregate by Entity and Time

In [16]:
time_ready = df_pred[df_pred["year_month"].notna()].copy()

entity_time_summary = (
    aggregate_sentiment(
        time_ready,
        group_cols=["canonical_entity", "final_type", "year_month"],
    )
    .sort_values(["final_type", "canonical_entity", "year_month"])
    .reset_index(drop=True)
)

export_table(entity_time_summary, ENTITY_TIME_SUMMARY_PARQUET, ENTITY_TIME_SUMMARY_CSV)
display(entity_time_summary.head(20))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_time_sentiment_summary.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/entity_time_sentiment_summary.csv


,canonical_entity,final_type,year_month,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index
0,01.ai,company,2024-05,1,1,1,0.889000,0.848588,0.069656,0.081755,0.848588,0,0,1,positive,0.778932,0.918245,0.495774,0.778932
1,01.ai,company,2025-01,2,2,2,0.765487,0.482876,0.050923,0.472700,0.476377,1,0,1,mixed_or_unclear,0.425454,0.527300,0.246465,0.425454
2,01AI_Yi,company,2023-11,1,1,1,0.807887,0.914405,0.056657,0.028938,0.914405,0,0,1,positive,0.857747,0.971062,0.577340,0.857747
3,0G Foundation,company,2025-06,1,1,1,0.610548,0.892482,0.048422,0.059096,0.892482,0,0,1,positive,0.844061,0.940904,0.550484,0.844061
4,0G Labs,company,2024-03,1,1,1,0.867815,0.926496,0.055858,0.017645,0.926496,0,0,1,positive,0.870638,0.982355,0.592832,0.870638
5,0G Labs,company,2025-08,1,1,1,0.822577,0.922027,0.051612,0.026361,0.922027,0,0,1,positive,0.870414,0.973639,0.587421,0.870414
6,0G Labs,company,2025-09,2,1,1,0.838620,0.924391,0.053179,0.022430,0.924391,0,0,2,positive,0.871212,0.977570,0.590333,0.871212
7,0G Labs,company,2025-10,1,1,1,0.912402,0.607910,0.051084,0.341007,0.607910,0,0,1,positive,0.556826,0.658993,0.254347,0.556826
8,0M Technology Solutions,company,2024-11,1,1,1,0.790306,0.912151,0.067379,0.020469,0.912151,0,0,1,positive,0.844772,0.979531,0.573565,0.844772
9,0min Meta,company,2023-05,1,1,1,0.615964,0.921492,0.060542,0.017966,0.921492,0,0,1,positive,0.860950,0.982034,0.586043,0.860950


## Aggregate by Domain

In [17]:
domain_summary = (
    aggregate_sentiment(
        df_pred,
        group_cols=["domain", "final_type"],
    )
    .sort_values(["n_docs", "n_rows"], ascending=[False, False])
    .reset_index(drop=True)
)

export_table(domain_summary, DOMAIN_SUMMARY_PARQUET, DOMAIN_SUMMARY_CSV)
display(domain_summary.head(20))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/domain_sentiment_summary.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/domain_sentiment_summary.csv


,domain,final_type,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index
0,einpresswire.com,company,11367,2730,1,0.835087,0.886782,0.068823,0.143313,0.787864,1439,113,9815,positive,0.719041,0.856687,4.874000,0.719041
1,einpresswire.com,person,7094,2688,1,0.881278,0.853285,0.090244,0.250673,0.659083,1751,260,5083,positive,0.568839,0.749327,3.366034,0.568839
2,menafn.com,person,4985,2439,1,0.882081,0.846241,0.090949,0.279486,0.629565,1448,175,3362,positive,0.538616,0.720514,3.026927,0.538616
3,menafn.com,company,8059,2313,1,0.829326,0.855992,0.063153,0.264981,0.671866,2174,51,5834,positive,0.608713,0.735019,3.466013,0.608713
4,prnewswire.com,company,8618,2143,1,0.828004,0.845992,0.060876,0.242144,0.696980,2088,26,6504,positive,0.636104,0.757856,3.697724,0.636104
5,einpresswire.com,technology,9384,2132,1,0.723005,0.872617,0.084949,0.145800,0.769252,1167,260,7957,positive,0.684303,0.854200,4.480604,0.684303
6,prnewswire.com,person,4321,2069,1,0.883214,0.840249,0.070458,0.282122,0.647420,1298,52,2971,positive,0.576963,0.717878,3.162458,0.576963
7,menafn.com,technology,5409,1618,1,0.722837,0.866195,0.083921,0.152335,0.763743,691,144,4574,positive,0.679822,0.847665,4.258320,0.679822
8,einpresswire.com,government_institution,3078,1376,1,0.733193,0.827094,0.100595,0.271652,0.627753,770,127,2181,positive,0.527158,0.728348,2.775090,0.527158
9,prnewswire.com,technology,3639,1363,1,0.702957,0.872175,0.074468,0.113156,0.812375,308,56,3275,positive,0.737907,0.886844,4.723634,0.737907


## Aggregate by Topic

In [18]:
topic_group_cols = [c for c in ["topic_id", "topic_label", "topic_name"] if c in df_pred.columns]

if "topic_id" in topic_group_cols:
    topic_ready = df_pred[df_pred["topic_id"].notna()].copy()

    topic_summary = (
        aggregate_sentiment(
            topic_ready,
            group_cols=topic_group_cols,
        )
        .sort_values(["n_docs", "n_rows"], ascending=[False, False])
        .reset_index(drop=True)
    )
else:
    topic_summary = pd.DataFrame()

topic_summary.to_parquet(TOPIC_SUMMARY_PARQUET, index=False)
topic_summary.to_csv(TOPIC_SUMMARY_CSV, index=False)
print("wrote:", TOPIC_SUMMARY_PARQUET)
print("wrote:", TOPIC_SUMMARY_CSV)
display(topic_summary.head(20))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/topic_sentiment_summary.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/topic_sentiment_summary.csv


""


## Coverage-Filtered Ranking Tables

In [19]:
company_display = entity_summary[
    (entity_summary["final_type"] == "company")
    & (entity_summary["n_docs"] >= MIN_DOCS_FOR_DISPLAY)
    & (entity_summary["n_rows"] >= MIN_ROWS_FOR_DISPLAY)
].copy()

technology_display = entity_summary[
    (entity_summary["final_type"] == "technology")
    & (entity_summary["n_docs"] >= MIN_DOCS_FOR_DISPLAY)
    & (entity_summary["n_rows"] >= MIN_ROWS_FOR_DISPLAY)
].copy()

top_positive_companies_display = company_display.sort_values(
    ["impact_score", "n_docs", "n_rows"],
    ascending=[False, False, False],
).head(TOP_DISPLAY_N).reset_index(drop=True)

top_negative_companies_display = company_display.sort_values(
    ["impact_score", "n_docs", "n_rows"],
    ascending=[True, False, False],
).head(TOP_DISPLAY_N).reset_index(drop=True)

top_positive_technologies_display = technology_display.sort_values(
    ["impact_score", "n_docs", "n_rows"],
    ascending=[False, False, False],
).head(TOP_DISPLAY_N).reset_index(drop=True)

top_negative_technologies_display = technology_display.sort_values(
    ["impact_score", "n_docs", "n_rows"],
    ascending=[True, False, False],
).head(TOP_DISPLAY_N).reset_index(drop=True)

top_positive_companies_display.to_csv(TOP_POSITIVE_COMPANIES_DISPLAY_CSV, index=False)
top_negative_companies_display.to_csv(TOP_NEGATIVE_COMPANIES_DISPLAY_CSV, index=False)
top_positive_technologies_display.to_csv(TOP_POSITIVE_TECHNOLOGIES_DISPLAY_CSV, index=False)
top_negative_technologies_display.to_csv(TOP_NEGATIVE_TECHNOLOGIES_DISPLAY_CSV, index=False)

print("wrote:", TOP_POSITIVE_COMPANIES_DISPLAY_CSV)
print("wrote:", TOP_NEGATIVE_COMPANIES_DISPLAY_CSV)
print("wrote:", TOP_POSITIVE_TECHNOLOGIES_DISPLAY_CSV)
print("wrote:", TOP_NEGATIVE_TECHNOLOGIES_DISPLAY_CSV)

print("\nCoverage-filtered top positive companies:")
display(
    top_positive_companies_display[
        [
            "canonical_entity",
            "n_docs",
            "n_rows",
            "hard_label_mode",
            "sentiment_index",
            "impact_score",
            "pred_confidence_mean",
        ]
    ]
)

print("\nCoverage-filtered top negative companies:")
display(
    top_negative_companies_display[
        [
            "canonical_entity",
            "n_docs",
            "n_rows",
            "hard_label_mode",
            "sentiment_index",
            "impact_score",
            "pred_confidence_mean",
        ]
    ]
)

print("\nCoverage-filtered top positive technologies:")
display(
    top_positive_technologies_display[
        [
            "canonical_entity",
            "n_docs",
            "n_rows",
            "hard_label_mode",
            "sentiment_index",
            "impact_score",
            "pred_confidence_mean",
        ]
    ]
)

print("\nCoverage-filtered top negative technologies:")
display(
    top_negative_technologies_display[
        [
            "canonical_entity",
            "n_docs",
            "n_rows",
            "hard_label_mode",
            "sentiment_index",
            "impact_score",
            "pred_confidence_mean",
        ]
    ]
)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/top_positive_companies_display.csv
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/top_negative_companies_display.csv
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/top_positive_technologies_display.csv
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/top_negative_technologies_display.csv

Coverage-filtered top positive companies:


,canonical_entity,n_docs,n_rows,hard_label_mode,sentiment_index,impact_score,pred_confidence_mean



Coverage-filtered top negative companies:


,canonical_entity,n_docs,n_rows,hard_label_mode,sentiment_index,impact_score,pred_confidence_mean



Coverage-filtered top positive technologies:


,canonical_entity,n_docs,n_rows,hard_label_mode,sentiment_index,impact_score,pred_confidence_mean
0,LLMs,15,15,positive,0.414331,0.942131,0.810735



Coverage-filtered top negative technologies:


,canonical_entity,n_docs,n_rows,hard_label_mode,sentiment_index,impact_score,pred_confidence_mean
0,LLMs,15,15,positive,0.414331,0.942131,0.810735


## Inference Report

In [20]:
report = {
    "n_row_predictions": int(len(df_pred)),
    "n_unique_entities": int(df_pred["canonical_entity"].nunique()),
    "n_unique_company_entities": int(df_pred.loc[df_pred["final_type"] == "company", "canonical_entity"].nunique()),
    "n_unique_technology_entities": int(df_pred.loc[df_pred["final_type"] == "technology", "canonical_entity"].nunique()),
    "rows_by_type": df_pred["final_type"].value_counts(dropna=False).to_dict(),
    "prediction_distribution": df_pred["pred_sentiment_label"].value_counts(dropna=False).to_dict(),
    "row_level_prob_means": {
        "prob_negative_mean": float(df_pred["prob_negative"].mean()),
        "prob_mixed_or_unclear_mean": float(df_pred["prob_mixed_or_unclear"].mean()),
        "prob_positive_mean": float(df_pred["prob_positive"].mean()),
    },
    "entity_summary_rows": int(len(entity_summary)),
    "entity_time_summary_rows": int(len(entity_time_summary)),
    "domain_summary_rows": int(len(domain_summary)),
    "topic_summary_rows": int(len(topic_summary)),
    "top_positive_companies": top_positive_companies_display[
        ["canonical_entity", "n_docs", "n_rows", "hard_label_mode", "sentiment_index", "impact_score"]
    ].to_dict(orient="records"),
    "top_negative_companies": top_negative_companies_display[
        ["canonical_entity", "n_docs", "n_rows", "hard_label_mode", "sentiment_index", "impact_score"]
    ].to_dict(orient="records"),
    "top_positive_technologies": top_positive_technologies_display[
        ["canonical_entity", "n_docs", "n_rows", "hard_label_mode", "sentiment_index", "impact_score"]
    ].to_dict(orient="records"),
    "top_negative_technologies": top_negative_technologies_display[
        ["canonical_entity", "n_docs", "n_rows", "hard_label_mode", "sentiment_index", "impact_score"]
    ].to_dict(orient="records"),
}

safe_json_dump(report, INFERENCE_REPORT_JSON)
print("wrote:", INFERENCE_REPORT_JSON)
print(report)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_inference/sentiment_inference_report.json
{'n_row_predictions': 560998, 'n_unique_entities': 259595, 'n_unique_company_entities': 49399, 'n_unique_technology_entities': 73246, 'rows_by_type': {'person': 208911, 'technology': 146631, 'company': 131873, 'government_institution': 73583}, 'prediction_distribution': {'positive': 330451, 'mixed_or_unclear': 172097, 'negative': 58450}, 'row_level_prob_means': {'prob_negative_mean': 0.15228982269763947, 'prob_mixed_or_unclear_mean': 0.3048800230026245, 'prob_positive_mean': 0.542830228805542}, 'entity_summary_rows': 264291, 'entity_time_summary_rows': 377732, 'domain_summary_rows': 14816, 'topic_summary_rows': 0, 'top_positive_companies': [], 'top_negative_companies': [], 'top_positive_technologies': [{'canonical_entity': 'LLMs', 'n_docs': 15, 'n_rows': 15, 'hard_label_mode': 'positive', 'sentiment_index': 0.41433054208755493, 'impact_score': 0.9421311845279645}], 'to

## Quick Readout

In [21]:
pred_preview = pd.read_parquet(ROW_LEVEL_PREDICTIONS_PARQUET)
entity_summary_preview = pd.read_parquet(ENTITY_SUMMARY_PARQUET)
entity_type_summary_preview = pd.read_parquet(ENTITY_TYPE_SUMMARY_PARQUET)
entity_time_summary_preview = pd.read_parquet(ENTITY_TIME_SUMMARY_PARQUET)
domain_summary_preview = pd.read_parquet(DOMAIN_SUMMARY_PARQUET)

print("row_level_predictions shape:", pred_preview.shape)
print("entity_summary shape:", entity_summary_preview.shape)
print("entity_type_summary shape:", entity_type_summary_preview.shape)
print("entity_time_summary shape:", entity_time_summary_preview.shape)
print("domain_summary shape:", domain_summary_preview.shape)

print("\nprediction distribution:")
display(pred_preview["pred_sentiment_label"].value_counts(dropna=False))

print("\nrows by final_type:")
display(pred_preview["final_type"].value_counts(dropna=False))

print("\nprediction confidence summary:")
display(pred_preview["pred_confidence"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]))

print("\nentity type sentiment summary:")
display(entity_type_summary_preview)

print("\ntop entities by document coverage:")
display(
    entity_summary_preview[
        (entity_summary_preview["n_docs"] >= 20) | (entity_summary_preview["n_rows"] >= 30)
    ]
    .sort_values(
        ["n_docs", "n_rows", "abs_sentiment_index", "pred_confidence_mean"],
        ascending=[False, False, False, False],
    )
    .head(20)
)

print("\nexample row-level predictions:")
display(
    pred_preview[
        [
            "canonical_entity",
            "final_type",
            "pred_sentiment_label",
            "pred_confidence",
            "domain",
            "title",
            "context_text",
        ]
    ].head(20)
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

row_level_predictions shape: (560998, 19)
entity_summary shape: (264291, 18)
entity_type_summary shape: (4, 17)
entity_time_summary shape: (377732, 19)
domain_summary shape: (14816, 18)

prediction distribution:


,count
pred_sentiment_label,
positive,330451
mixed_or_unclear,172097
negative,58450



rows by final_type:


,count
final_type,
person,208911
technology,146631
company,131873
government_institution,73583



prediction confidence summary:


...

,final_type,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index
0,person,208911,98601,4640,0.868538,0.845963,0.186766,0.384602,0.428632,84015,29928,94968,positive,0.241866,0.615398,1.711533,0.241866
1,company,131873,57163,3102,0.801551,0.871440,0.100937,0.215452,0.683612,27313,6252,98308,positive,0.582675,0.784548,5.007331,0.582675
2,technology,146631,55619,3422,0.713391,0.859716,0.130310,0.239508,0.630182,33321,11711,101599,positive,0.499872,0.760492,4.153617,0.499872
3,government_institution,73583,40753,3652,0.741925,0.838197,0.190242,0.369079,0.440680,27448,10559,35576,positive,0.250438,0.630921,1.677289,0.250438



top entities by document coverage:


,canonical_entity,final_type,n_rows,n_docs,n_domains,mention_confidence_mean,pred_confidence_mean,prob_negative_mean,prob_mixed_or_unclear_mean,prob_positive_mean,mixed_or_unclear,negative,positive,hard_label_mode,sentiment_index,directional_confidence,impact_score,abs_sentiment_index



example row-level predictions:


,canonical_entity,final_type,pred_sentiment_label,pred_confidence,domain,title,context_text
0,SABI AI Corp,company,positive,0.923650,uppermichiganssource.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
1,SABI AI Corp,company,positive,0.923455,wdtv.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
2,SABI AI Corp,company,positive,0.924199,walb.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
3,SABI AI Corp,company,positive,0.923650,uppermichiganssource.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
4,SABI AI Corp,company,positive,0.923455,wdtv.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 1 hour a..."
5,SABI AI Corp,company,positive,0.924199,walb.com,Skin Inc Supplement Bar in Collaboration with ...,"Mar. 21, 2022 at 7:00 AM EDT|Updated: 58 minut..."
6,01.ai,company,positive,0.848588,irishtimes.com,Chinese AI start-ups power ahead in the race t...,May 9 2024 - 05:00Four Chinese generative arti...
7,01.ai,company,mixed_or_unclear,0.481885,nbcmiami.com,How China’s new AI model DeepSeek is threateni...,1 hour ago The No. 1 mistake people make when ...
8,01.ai,company,positive,0.483866,nbcdfw.com,How China’s new AI model DeepSeek is threateni...,45 mins ago Here's what CEOs are saying about ...
9,01AI_Yi,company,positive,0.914405,thehindu.com,AI scientist Kai-Fu Lee’s startup releases ope...,Indian engineering student builds AI model for...
